In [1]:
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
import os
import pandas as pd
from tqdm import tqdm
import random

In [2]:
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7)
mp_drawing = mp.solutions.drawing_utils

num_classes = 26
img_per_class = 400

train_len = num_classes * img_per_class
train_dir = 'data/asl_long_dataset/asl_alphabet_train'

np.random.seed(42)
random.seed(42)

def get_data(folder):
    X = np.empty((train_len, 42), dtype=np.float32)
    z_depth = np.empty((train_len,), dtype=np.float32)
    y = np.empty((train_len,), dtype=int)

    label_map = {chr(65+i): i for i in range(26)}
    
    cnt = 0
    for letter in tqdm(label_map.keys(), desc="Extraction des Landmarks"):
        folder_path = os.path.join(folder, letter)
        cnt_letter = 0
        img_indice = np.random.randint(1, 3001, 3000)
        for img_num in img_indice:
            img_path = os.path.join(folder_path, letter+str(img_num)+".jpg")
            img_file = cv2.imread(img_path)
            image_rgb = cv2.cvtColor(img_file, cv2.COLOR_BGR2RGB)
            results = hands.process(image_rgb)
            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    points_coord = np.array([[lm.x, lm.y] for lm in hand_landmarks.landmark])
                    X[cnt] = points_coord.reshape((42,))
                    z_depth[cnt] =  np.array([lm.z for lm in hand_landmarks.landmark]).mean()
                    
                    y[cnt] = label_map[letter]
                    cnt += 1
                    cnt_letter +=1
                    break
                    
            if cnt_letter >= img_per_class:
                break

    print(f"\n Extraction terminée : {cnt} images traitées.")
    return X[:cnt], y[:cnt], z_depth[:cnt]

X, y, z = get_data(train_dir)

Extraction des Landmarks: 100%|████████████████████████████████████████████████████████| 26/26 [09:51<00:00, 22.74s/it]


 Extraction terminée : 10400 images traitées.


In [3]:
df = pd.DataFrame(X, columns=[f"p{i}" for i in range(42)])
df["depth"] = z.astype(float)
df["label"] = y.astype(int)

df.to_csv("data/asl_dataset.csv", index=False)
